# Data Preprocessing | Encoder Part

`We clean and prepare data before we train a model.`

## 1. Data Collection

We get data from many places:

- (CSV - Excel - JSON - XML - etc.) files
- Databases
- APIs
- scraping websites
- sensors
- user input (forms, surveys, etc.)
- etc.

In [59]:
# Importing
import pandas as pd
import numpy as np

In [60]:
# Load the students data
students = pd.read_csv ("../data/students.csv")

# Show the first 5 rows
students.head ()

,Name,Age,Salary,City,Education,Target
0,Mohamed,43.0,14080,Dubai,Master,0
1,Zaid,39.0,8393,Cairo,PhD,0
2,Ahmed Mohamed,29.0,13627,Dubai,High School,0
3,Zaid,NaN,11792,Dubai,Bachelor,0
4,Zaid,34.0,13555,Dubai,PhD,0


## 2. Data Preparation

After we load the data, we prepare it.

Common problems are:

- Missing values (empty cells)
- Duplicate rows (the same row twice)
- Wrong data types
- Outliers (very strange values)

### 2.1 Check the Data

Before we change anything, we look at the data.

We want to know:

- How many rows and columns are there?
- What are the column names?
- Are there missing values?
- Are there duplicate rows?

In [61]:
# Number of rows and columns
print ("Shape:", students.shape)

# Column names and data types
students.info ()


Shape: (21, 6)
<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Name       21 non-null     str    
 1   Age        20 non-null     float64
 2   Salary     21 non-null     int64  
 3   City       21 non-null     str    
 4   Education  21 non-null     str    
 5   Target     21 non-null     int64  
dtypes: float64(1), int64(2), str(3)
memory usage: 1.5 KB


In [62]:
# Quick numbers for numerical columns
students.describe ()

,Age,Salary,Target
count,20.000000,21.000000,21.000000
mean,34.350000,11171.190476,0.238095
std,7.499298,2824.806500,0.436436
min,20.000000,3775.000000,0.000000
25%,29.000000,10041.000000,0.000000
50%,35.000000,11792.000000,0.000000
75%,40.500000,13253.000000,0.000000
max,44.000000,14080.000000,1.000000


In [63]:
# Count missing values in each column
students.isnull ().sum ()


Name         0
Age          1
Salary       0
City         0
Education    0
Target       0
dtype: int64

In [64]:
# Count duplicate rows
print("Duplicate rows:", students.duplicated ().sum ())
# students[students.duplicated(keep= "last")]
# students[students.duplicated(keep= "first")]
students [students.duplicated (keep= False)]

Duplicate rows: 1


,Name,Age,Salary,City,Education,Target
5,Ahmed Zaid,44.0,13253,Dubai,High School,0
20,Ahmed Zaid,44.0,13253,Dubai,High School,0


**What did we find?**

- `Age` has missing values.
- Some rows are duplicated.

Next, we learn how to fill missing values. After that, we remove duplicates.

### 2.2 Handling Missing Values

A **missing value** is an empty cell. In pandas, it often looks like `NaN`.

**Why is this a problem?**

- Many models cannot work with empty cells.
- Missing values can make results wrong.

**How do we check?**

```python
df.isnull().sum()
```

**What should we do first?**

Do not fill missing values right away.

First ask:

- Why is the value missing?
- Is the column useful?
- Is the column numerical or categorical?
- How many missing values are there?

Then choose a method.

In [65]:
# Small example with missing values
df_missing = pd.DataFrame({
    "Age": [25, np.nan, 30, 22, np.nan, 40, 28],
    "Salary": [3000, 4500, np.nan, 2800, 5000, np.nan, 3200],
    "City": ["Cairo", "Dubai", "Cairo", np.nan, "Paris", "Dubai", "Cairo"],
    "Passed": [1, 0, 1, 0, 1, 1, 0],
})

print("Missing values:")
print(df_missing.isnull().sum())
df_missing


Missing values:
Age       2
Salary    2
City      1
Passed    0
dtype: int64


,Age,Salary,City,Passed
0,25.0,3000.0,Cairo,1
1,NaN,4500.0,Dubai,0
2,30.0,NaN,Cairo,1
3,22.0,2800.0,NaN,0
4,NaN,5000.0,Paris,1
5,40.0,NaN,Dubai,1
6,28.0,3200.0,Cairo,0


#### Mean Imputation

**What is it?**

We fill missing **numbers** with the **mean** (average).

**Why use it?**

It is simple. It keeps the column size the same.

**When is it useful?**

- The column is numerical.
- The data does not have strong outliers.
- The data is normally distributed.

In [66]:
# Copy the example so we do not change the original
df_mean = df_missing.copy()

# Mean of Age and Salary
age_mean = df_mean["Age"].mean()
salary_mean = df_mean["Salary"].mean()

print("Age mean:", round(age_mean, 2))
print("Salary mean:", round(salary_mean, 2))

# Fill missing numbers with the mean
df_mean["Age"] = df_mean["Age"].fillna(age_mean)
df_mean["Salary"] = df_mean["Salary"].fillna(salary_mean)

print()
print("Missing values after mean fill:")
print(df_mean.isnull().sum())
df_mean


Age mean: 29.0
Salary mean: 3700.0

Missing values after mean fill:
Age       0
Salary    0
City      1
Passed    0
dtype: int64


,Age,Salary,City,Passed
0,25.0,3000.0,Cairo,1
1,29.0,4500.0,Dubai,0
2,30.0,3700.0,Cairo,1
3,22.0,2800.0,NaN,0
4,29.0,5000.0,Paris,1
5,40.0,3700.0,Dubai,1
6,28.0,3200.0,Cairo,0


We can also use `SimpleImputer` from scikit-learn.

This is useful later, because we can fit it on training data only.

In [67]:
from sklearn.impute import SimpleImputer

df_mean_sk = df_missing.copy()

mean_imputer = SimpleImputer (strategy="mean")
df_mean_sk[["Age", "Salary"]] = mean_imputer.fit_transform(
    df_mean_sk[["Age", "Salary"]]
)

df_mean_sk


,Age,Salary,City,Passed
0,25.0,3000.0,Cairo,1
1,29.0,4500.0,Dubai,0
2,30.0,3700.0,Cairo,1
3,22.0,2800.0,NaN,0
4,29.0,5000.0,Paris,1
5,40.0,3700.0,Dubai,1
6,28.0,3200.0,Cairo,0


#### Median Imputation

**What is it?**

We fill missing **numbers** with the **median**.

The median is the middle value after we sort the numbers.

**Why use it?**

It is also simple. It is more stable than the mean when outliers exist.

**When is it useful?**

- The column is numerical.
- The data may have outliers.

**Note:** The median is more resistant to outliers than the mean.

In [68]:
df_median = df_missing.copy()

# Median of Age and Salary
age_median = df_median["Age"].median()
salary_median = df_median["Salary"].median()

print("Age median:", age_median)
print("Salary median:", salary_median)

# Fill missing numbers with the median
df_median["Age"] = df_median["Age"].fillna(age_median)
df_median["Salary"] = df_median["Salary"].fillna(salary_median)

print()
print("Missing values after median fill:")
print(df_median.isnull().sum())
df_median


Age median: 28.0
Salary median: 3200.0

Missing values after median fill:
Age       0
Salary    0
City      1
Passed    0
dtype: int64


,Age,Salary,City,Passed
0,25.0,3000.0,Cairo,1
1,28.0,4500.0,Dubai,0
2,30.0,3200.0,Cairo,1
3,22.0,2800.0,NaN,0
4,28.0,5000.0,Paris,1
5,40.0,3200.0,Dubai,1
6,28.0,3200.0,Cairo,0


In [69]:
df_median_sk = df_missing.copy()

median_imputer = SimpleImputer(strategy="median")
df_median_sk[["Age", "Salary"]] = median_imputer.fit_transform(
    df_median_sk[["Age", "Salary"]]
)

df_median_sk


,Age,Salary,City,Passed
0,25.0,3000.0,Cairo,1
1,28.0,4500.0,Dubai,0
2,30.0,3200.0,Cairo,1
3,22.0,2800.0,NaN,0
4,28.0,5000.0,Paris,1
5,40.0,3200.0,Dubai,1
6,28.0,3200.0,Cairo,0


#### Mode Imputation

**What is it?**

We fill missing **categories** with the **mode**.

The mode is the most frequent value. For example, if most people live in Cairo, we can fill empty City values with Cairo.

**Why use it?**

Mean and median do not work well for text categories like City.

**When is it useful?**

- The column is categorical.
- One category is clearly the most common.

**Note:** The mode is the most frequent value.

In [70]:
df_mode = df_missing.copy()

# Most common city
city_mode = df_mode["City"].mode()[0]
print("City mode:", city_mode)

# Fill missing City values with the mode
df_mode["City"] = df_mode["City"].fillna(city_mode)

print()
print("Missing values after mode fill:")
print(df_mode.isnull().sum())
df_mode


City mode: Cairo

Missing values after mode fill:
Age       2
Salary    2
City      0
Passed    0
dtype: int64


,Age,Salary,City,Passed
0,25.0,3000.0,Cairo,1
1,NaN,4500.0,Dubai,0
2,30.0,NaN,Cairo,1
3,22.0,2800.0,Cairo,0
4,NaN,5000.0,Paris,1
5,40.0,NaN,Dubai,1
6,28.0,3200.0,Cairo,0


In [71]:
df_mode_sk = df_missing.copy()

mode_imputer = SimpleImputer(strategy="most_frequent")
df_mode_sk[["City"]] = mode_imputer.fit_transform(df_mode_sk[["City"]])

df_mode_sk


,Age,Salary,City,Passed
0,25.0,3000.0,Cairo,1
1,NaN,4500.0,Dubai,0
2,30.0,NaN,Cairo,1
3,22.0,2800.0,Cairo,0
4,NaN,5000.0,Paris,1
5,40.0,NaN,Dubai,1
6,28.0,3200.0,Cairo,0


#### KNN Imputation

**What is it?**

KNN looks at similar rows. Then it uses their values to estimate the missing value.

If two people have a similar salary, their ages may also be similar.

**Why use it?**

It uses more than one column. It can be better than using only the mean or median.

**When is it useful?**

- The missing values are numerical.
- Other columns can help guess the missing value.

**Note:** `KNNImputer` works with numbers. We do not use it on City in this example.<br>
**Note:** KNN can be slow on large datasets. It is better for small datasets. <br>
**Note:** It is preferable to scale the data before using KNN imputation, as it relies on distance metrics. <br>

In [72]:
from sklearn.impute import KNNImputer

df_knn = df_missing.copy()

display (df_knn)

knn_imputer = KNNImputer(n_neighbors=2)
df_knn[["Age", "Salary"]] = knn_imputer.fit_transform(
    df_knn[["Age", "Salary"]]
)

print("Missing values after KNN fill:")
print(df_knn.isnull().sum())
df_knn


,Age,Salary,City,Passed
0,25.0,3000.0,Cairo,1
1,NaN,4500.0,Dubai,0
2,30.0,NaN,Cairo,1
3,22.0,2800.0,NaN,0
4,NaN,5000.0,Paris,1
5,40.0,NaN,Dubai,1
6,28.0,3200.0,Cairo,0


Missing values after KNN fill:
Age       0
Salary    0
City      1
Passed    0
dtype: int64


,Age,Salary,City,Passed
0,25.0,3000.0,Cairo,1
1,26.5,4500.0,Dubai,0
2,30.0,3100.0,Cairo,1
3,22.0,2800.0,NaN,0
4,26.5,5000.0,Paris,1
5,40.0,3100.0,Dubai,1
6,28.0,3200.0,Cairo,0


#### MICE / Iterative Imputation

**What is it?**

MICE means **Multiple Imputation by Chained Equations**.

The simple idea is:

1. Use other columns to guess a missing value.
2. Repeat this for each column with missing values.
3. Repeat the process a few times to improve the guesses.

scikit-learn does not have a tool named `MICE`. It has `IterativeImputer`.

`IterativeImputer` is an iterative method. The idea is similar to MICE.

**Why use it?**

It can use relationships between columns.

**When is it useful?**

- More than one numerical column has missing values.
- The columns are related.

**Note:** This tool is experimental in scikit-learn. We must enable it first.

In [73]:
# This import turns on IterativeImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

df_mice = df_missing.copy()

mice_imputer = IterativeImputer(max_iter=50, random_state=42)
df_mice[["Age", "Salary"]] = mice_imputer.fit_transform(
    df_mice[["Age", "Salary"]]
)

print("Missing values after iterative fill:")
print(df_mice.isnull().sum())
df_mice.head()


Missing values after iterative fill:
Age       0
Salary    0
City      1
Passed    0
dtype: int64


,Age,Salary,City,Passed
0,25.000000,3000.000000,Cairo,1
1,46.278699,4500.000000,Dubai,0
2,30.000000,3352.860224,Cairo,1
3,22.000000,2800.000000,NaN,0
4,53.379124,5000.000000,Paris,1


#### Avoid Data Leakage When Filling Missing Values

**Data leakage** means the model sees information from the test data too early.

This can make the test score look better than it really is.

**The rule**

> Fit the imputer on the training data only.

Do **not** calculate the mean, median, or mode on the full dataset before splitting.

- `fit` = learn from training data
- `transform` = apply the same rule to train and test

A `Pipeline` can do this in a safe way.

In [74]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Use the small example
X = df_missing[["Age", "Salary"]]
y = df_missing["Passed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Learn from training data only
imputer = SimpleImputer(strategy="median")
X_train_filled = imputer.fit_transform(X_train)
X_test_filled = imputer.transform(X_test)

print("Train missing after fill:", np.isnan(X_train_filled).sum())
print("Test missing after fill:", np.isnan(X_test_filled).sum())
print("Median learned from train:", imputer.statistics_)


Train missing after fill: 0
Test missing after fill: 0
Median learned from train: [  28. 3200.]


In [75]:
# A Pipeline also fits on training data only
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

X_train_pipe = pipe.fit_transform(X_train)
X_test_pipe = pipe.transform(X_test)

print("Train shape:", X_train_pipe.shape)
print("Test shape:", X_test_pipe.shape)


Train shape: (4, 2)
Test shape: (3, 2)


### 2.3 Removing Duplicates

A **duplicate** is the same row more than once.

Duplicates can make the model see the same example too many times.

First we check. Then we remove them.

In [76]:
# Work on a copy of the students data
students_clean = students.copy()

print("Rows before:", len(students_clean))
print("Duplicate rows:", students_clean.duplicated().sum())

# Remove duplicate rows
students_clean = students_clean.drop_duplicates()

print("Rows after:", len(students_clean))
students_clean.head()


Rows before: 21
Duplicate rows: 1
Rows after: 20


,Name,Age,Salary,City,Education,Target
0,Mohamed,43.0,14080,Dubai,Master,0
1,Zaid,39.0,8393,Cairo,PhD,0
2,Ahmed Mohamed,29.0,13627,Dubai,High School,0
3,Zaid,NaN,11792,Dubai,Bachelor,0
4,Zaid,34.0,13555,Dubai,PhD,0


On the students data, we can also fill `Age` after we understand the methods.

In a real project, we should split first, then fill missing values on the training data only.

Here we only show the pandas method on this small file.

In [77]:
print("Missing Age before:", students_clean["Age"].isnull().sum())

# Median is a safer default than mean if outliers may exist
students_clean["Age"] = students_clean["Age"].fillna(
    students_clean["Age"].median()
)

print("Missing Age after:", students_clean["Age"].isnull().sum())
students_clean.head()


Missing Age before: 1
Missing Age after: 0


,Name,Age,Salary,City,Education,Target
0,Mohamed,43.0,14080,Dubai,Master,0
1,Zaid,39.0,8393,Cairo,PhD,0
2,Ahmed Mohamed,29.0,13627,Dubai,High School,0
3,Zaid,34.0,11792,Dubai,Bachelor,0
4,Zaid,34.0,13555,Dubai,PhD,0


## 3. Handling Outliers

An **outlier** is a value that is very different from the other values.

Example: most house prices are around 200,000, but one house is 950,000.

Outliers can:

- Change the mean
- Change scaling
- Confuse some models

We will study two methods:

1. IQR
2. Z-Score

We use a small house price dataset.

In [78]:
houses = pd.read_csv("../data/house_prices.csv")
houses


,Area,Bedrooms,Age,Price
0,1200,2,10,145000
1,1500,3,5,210000
2,1700,3,8,230000
3,2000,4,12,275000
4,850,2,30,95000
5,2200,4,7,310000
6,2500,5,4,360000
7,1800,3,15,225000
8,1300,2,20,130000
9,1600,3,9,205000


### 3.1 IQR Method

IQR means **Interquartile Range**.

**Steps**

1. Q1 is the 25th percentile.
2. Q3 is the 75th percentile.
3. IQR = Q3 - Q1
4. Lower bound = Q1 - 1.5 × IQR
5. Upper bound = Q3 + 1.5 × IQR

A value below the lower bound or above the upper bound is an outlier.

**Example**

If Q1 = 20 and Q3 = 40:

- IQR = 20
- Lower bound = -10
- Upper bound = 70

So 100 is an outlier. 25 is normal.

**Note:** 1.5 is a common choice.

- 1.5 × IQR finds normal and moderate outliers.
- 3 × IQR finds only extreme outliers.

In [79]:
# Calculate Q1, Q3, and IQR
Q1 = houses.quantile(0.25)
Q3 = houses.quantile(0.75)
IQR = Q3 - Q1

outlier_condition = (houses < (Q1 - 1.5 * IQR)) | (houses > (Q3 + 1.5 * IQR))
outlier_condition


,Area,Bedrooms,Age,Price
0,False,False,False,False
1,False,False,False,False
2,False,False,False,False
3,False,False,False,False
4,False,False,False,False
5,False,False,False,False
6,False,False,False,False
7,False,False,False,False
8,False,False,False,False
9,False,False,False,False


In [80]:
# Rows with at least one outlier
outliers_iqr = houses[outlier_condition.any(axis=1)]
print("Outliers with IQR:")
outliers_iqr


Outliers with IQR:


,Area,Bedrooms,Age,Price
10,4000,5,3,780000
12,600,1,40,65000
13,5000,6,1,950000
14,700,1,50,55000


In [81]:
# Keep rows with no outliers
houses_iqr_clean = houses[~outlier_condition.any(axis=1)]

print("Original rows:", len(houses))
print("Rows after IQR:", len(houses_iqr_clean))
houses_iqr_clean


Original rows: 15
Rows after IQR: 11


,Area,Bedrooms,Age,Price
0,1200,2,10,145000
1,1500,3,5,210000
2,1700,3,8,230000
3,2000,4,12,275000
4,850,2,30,95000
5,2200,4,7,310000
6,2500,5,4,360000
7,1800,3,15,225000
8,1300,2,20,130000
9,1600,3,9,205000


In [ ]:
# Func to calc lower and upper bounds
def calculate_iqr_bounds (data : pd.DataFrame | pd.Series):
    """
    Calculate the IQR and lower and upper bounds.

    Parameters
    ----------
    data : pd.DataFrame or pd.Series
        The input data.
        If a DataFrame is passed, the calculation is performed
        separately for each column (each column is a Series).

    Returns
    -------
    tuple
        A tuple containing:
        - IQR
        - Lower bound
        - Upper bound
    """
    
    q1 = data.quantile (0.25)
    q3 = data.quantile (0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    return iqr, lower_bound, upper_bound


### 3.2 Z-Score Method

A **Z-score** shows how far a value is from the mean.

$$Z = \frac{X - \mu}{\sigma}$$

- X is the value
- μ is the mean
- σ is the standard deviation

A common rule is:

- |Z| ≤ 3 → normal
- |Z| > 3 → possible outlier

We use the absolute value because an outlier can be very high or very low.

In [83]:
print ("Mean:")
print (houses.mean())
print ()
print ("Standard deviation:")
print (houses.std())

Mean:
Area          1996.666667
Bedrooms         3.200000
Age             14.400000
Price       303666.666667
dtype: float64

Standard deviation:
Area          1222.770079
Bedrooms         1.473577
Age             14.657031
Price       259570.157504
dtype: float64


In [84]:
# Z-score for every value
z_scores = (houses - houses.mean()) / houses.std()
z_scores

,Area,Bedrooms,Age,Price
0,-0.651526,-0.814345,-0.300197,-0.611267
1,-0.406182,-0.135724,-0.641330,-0.360853
2,-0.242619,-0.135724,-0.436650,-0.283803
3,0.002726,0.542897,-0.163744,-0.110439
4,-0.937761,-0.814345,1.064336,-0.803893
5,0.166289,0.542897,-0.504877,0.024399
6,0.411634,1.221518,-0.709557,0.217025
7,-0.160837,-0.135724,0.040936,-0.303065
8,-0.569745,-0.814345,0.382069,-0.669055
9,-0.324400,-0.135724,-0.368424,-0.380116


In [85]:
threshold = 2

outliers_z = houses[(np.abs(z_scores) > threshold).any(axis=1)]
print("Outliers with Z-score:")
outliers_z


Outliers with Z-score:


,Area,Bedrooms,Age,Price
13,5000,6,1,950000
14,700,1,50,55000


In [86]:
# Keep rows where all |Z| values are below the threshold
houses_z_clean = houses[(np.abs(z_scores) < threshold).all(axis=1)]
print("Original rows:", len(houses))
print("Rows after Z-score:", len(houses_z_clean))
houses_z_clean


Original rows: 15
Rows after Z-score: 13


,Area,Bedrooms,Age,Price
0,1200,2,10,145000
1,1500,3,5,210000
2,1700,3,8,230000
3,2000,4,12,275000
4,850,2,30,95000
5,2200,4,7,310000
6,2500,5,4,360000
7,1800,3,15,225000
8,1300,2,20,130000
9,1600,3,9,205000


In [ ]:
# Func to calc z score

def calc_z_score (data : pd.DataFrame | pd.Series, is_sample : bool = False):
    """
    Calculate z score for a DataFrame or Series.
    """
    mean = data.mean ()
    std = data.std (ddof= int (is_sample))
    z = (data - mean) / std
    return z

**IQR vs Z-Score**

- IQR uses the middle part of the data. It works even if the data is not a bell shape.
- Z-score uses the mean and standard deviation. It works best when the data is close to a bell shape.

On this small house dataset, IQR finds some outliers. Z-score with |Z| > 3 finds none. This can happen when the dataset is small and the standard deviation is large.

Do not always delete outliers. First try to understand them.

## 4. Data Encoding

Most models understand numbers, not text.

**Encoding** means we turn categories into numbers.

Examples of categories:

- City
- Education
- Gender

There are two main types:

1. **Ordinal** → there is a real order.
2. **Nominal** → there is no real order.

### 4.1 Ordinal Encoding

Ordinal categories have an order.

Example — Education:

```
High School < Bachelor < Master < PhD
```

Other examples:

- Size: Small < Medium < Large
- Level: Beginner < Intermediate < Advanced

We should keep this order when we use numbers.

A common mistake is to use `LabelEncoder` on ordinal data.

`LabelEncoder` does not know the real order. It often uses alphabetical order.

In [87]:
from sklearn.preprocessing import LabelEncoder

edu = pd.DataFrame({
    "Education": ["High School", "Bachelor", "Master", "PhD", "Bachelor"]
})

# Wrong for ordinal data: alphabetical order
le = LabelEncoder()
edu["Education_Wrong"] = le.fit_transform(edu["Education"])
edu


,Education,Education_Wrong
0,High School,1
1,Bachelor,0
2,Master,2
3,PhD,3
4,Bachelor,0


The real order is:

```
High School < Bachelor < Master < PhD
```

But `LabelEncoder` may give:

```
Bachelor < High School < Master < PhD
```

This is wrong.

A simple correct way is a dictionary (a mapping).

In [88]:
# Correct order for Education
education_mapping = {
    "High School": 0,
    "Bachelor": 1,
    "Master": 2,
    "PhD": 3,
}

edu["Education_Encoded"] = edu["Education"].map(education_mapping)
edu


,Education,Education_Wrong,Education_Encoded
0,High School,1,0
1,Bachelor,0,1
2,Master,2,2
3,PhD,3,3
4,Bachelor,0,1


Now the numbers keep the real order:

```
High School -> 0
Bachelor    -> 1
Master      -> 2
PhD         -> 3
```

**Important:** These numbers are labels, not scores.

`PhD = 3` does **not** mean PhD is three times better than `Bachelor = 1`.

### 4.2 Nominal Encoding

Nominal categories have **no** natural order.

Example — City:

```
Cairo, Dubai, Paris
```

There is no real order like:

```
Cairo < Dubai < Paris
```

If we give them 0, 1, 2, we create a fake order. This can confuse the model.

For nominal features, we use **One-Hot Encoding**.

### 4.3 One-Hot Encoding

One-Hot Encoding makes a new column for each category.

Each new column is 0 or 1.

Original data:

| City |
|------|
| Cairo |
| Dubai |
| Paris |
| Cairo |

After One-Hot Encoding:

| City_Cairo | City_Dubai | City_Paris |
|------------|------------|------------|
| 1 | 0 | 0 |
| 0 | 1 | 0 |
| 0 | 0 | 1 |
| 1 | 0 | 0 |

In [89]:
city = pd.DataFrame({
    "City": ["Cairo", "Dubai", "Paris", "Cairo"]
})

city_encoded = pd.get_dummies(city, columns=["City"], dtype=int)
city_encoded


,City_Cairo,City_Dubai,City_Paris
0,1,0,0
1,0,1,0
2,0,0,1
3,1,0,0


**Why use One-Hot Encoding?**

- It does not create a fake order.
- Each city is treated the same way.

**Do not encode every text column.**

If a column does not help the model, remove it:

```python
df = df.drop("City", axis=1)
```

**How to choose**

> Does this feature have a natural order?

- **Yes** → Ordinal Encoding
- **No** → One-Hot Encoding

| Feature type | Has order? | Example | Method |
|--------------|------------|---------|--------|
| Ordinal | Yes | Education | Mapping / ordinal encoding |
| Nominal | No | City | One-Hot Encoding |
| Binary | Two values | Yes / No | 0 / 1 |

## 5. Feature Scaling

**Feature Scaling** changes the values of features to a similar scale.

**Why?**

A feature with big numbers can control a feature with small numbers.

Example:

| Age | Income |
|-----|--------|
| 20 | 30,000 |
| 30 | 50,000 |
| 40 | 80,000 |

Income has much bigger numbers than Age.

Models that use **distance** (like KNN) can then focus too much on Income.

Scaling helps with:

- Fair feature size
- Distance models (KNN, K-Means, SVM)
- Faster and more stable training for some models

### 5.1 Min-Max Scaling

Min-Max Scaling puts values in a fixed range, usually 0 to 1.

$$X' = \frac{X - X_{min}}{X_{max} - X_{min}}$$

**Example**

Ages: 4, 10, 17

- min = 4
- max = 17
- For 10: (10 - 4) / (17 - 4) = 0.46

In [90]:
from sklearn.preprocessing import MinMaxScaler

scale_df = pd.DataFrame({
    "Age": [4, 10, 17],
    "Worth": [48000, 60000, 83000],
})

minmax = MinMaxScaler()
scale_df[["Age_minmax", "Worth_minmax"]] = minmax.fit_transform(
    scale_df[["Age", "Worth"]]
)
scale_df


,Age,Worth,Age_minmax,Worth_minmax
0,4,48000,0.000000,0.000000
1,10,60000,0.461538,0.342857
2,17,83000,1.000000,1.000000


**When to use it**

- You want a fixed range like 0 to 1.
- The data does not have big outliers.

**Note:** Min-Max Scaling is sensitive to outliers. One very big value can squeeze the other values.

### 5.2 Standardization

Standardization changes a feature so that:

- Mean ≈ 0
- Standard deviation ≈ 1

$$Z = \frac{X - \mu}{\sigma}$$

This is also called Z-score scaling.

In [91]:
from sklearn.preprocessing import StandardScaler

std_df = pd.DataFrame({
    "Age": [4, 10, 17],
    "Worth": [48000, 60000, 83000],
})

standard = StandardScaler()
std_df[["Age_std", "Worth_std"]] = standard.fit_transform(
    std_df[["Age", "Worth"]]
)
std_df


,Age,Worth,Age_std,Worth_std
0,4,48000,-1.192166,-1.078822
1,10,60000,-0.062746,-0.252490
2,17,83000,1.254912,1.331312


**When to use it**

It is useful for:

- Linear Regression
- Logistic Regression
- SVM
- PCA
- Neural Networks
- KNN
- K-Means

**Note:** Standardization does not remove outliers. Outliers can still change the mean and the standard deviation.

### 5.3 Robust Scaling

Robust Scaling reduces the effect of outliers.

It uses:

- Median
- IQR

$$X' = \frac{X - Median}{IQR}$$

If one income is 200,000 and the others are around 30,000, the mean can change a lot. The median changes less.

In [92]:
from sklearn.preprocessing import RobustScaler

robust_df = pd.DataFrame({
    "Age": [20, 25, 60],
    "Income": [32000, 35000, 200000],
})

robust = RobustScaler()
robust_df[["Age_robust", "Income_robust"]] = robust.fit_transform(
    robust_df[["Age", "Income"]]
)
robust_df


,Age,Income,Age_robust,Income_robust
0,20,32000,-0.25,-0.035714
1,25,35000,0.00,0.000000
2,60,200000,1.75,1.964286


### 5.4 Comparing Scaling Methods

| Method | Uses | Result | Outliers |
|--------|------|--------|----------|
| Min-Max | Min and Max | Usually 0 to 1 | High effect |
| StandardScaler | Mean and Std | Mean 0, Std 1 | High effect |
| RobustScaler | Median and IQR | No fixed range | Lower effect |

**Which one?**

- **Min-Max**: no strong outliers, and you want 0 to 1.
- **StandardScaler**: different scales, no strong outliers.
- **RobustScaler**: strong outliers are present.

### 5.5 When Scaling Is Not Needed

Not every model needs scaling.

Ask this question:

> Does the model care about the size of the numbers?

**Usually need scaling**

- KNN, K-Means, DBSCAN
- SVM
- Logistic Regression
- Neural Networks
- PCA

**Usually do not need scaling**

Tree models split with rules like `Age < 30`. Changing the scale does not change the order.

- Decision Tree
- Random Forest
- Gradient Boosting
- XGBoost

### 5.6 Scaling and Data Leakage

Never fit the scaler on the test data.

Wrong:

```python
X_scaled = scaler.fit_transform(X)
X_train, X_test = train_test_split(X_scaled, test_size=0.2)
```

This is wrong because the scaler already saw the test data.

Correct:

1. Split the data.
2. Fit the scaler on the training data only.
3. Transform train and test with the same scaler.

In [93]:
from sklearn.linear_model import LogisticRegression

# Small numerical example for scaling
X_scale = pd.DataFrame({
    "Age": [18, 19, 20, 21, 22, 23, 24, 25, 26, 27],
    "Hours": [1, 2, 3, 4, 5, 6, 7, 8, 2, 9],
})
y_scale = pd.Series([0, 0, 0, 0, 1, 1, 1, 1, 0, 1], name="Passed")

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_scale, y_scale, test_size=0.3, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_s)
X_test_scaled = scaler.transform(X_test_s)

print("Train mean after scaling:", X_train_scaled.mean(axis=0).round(3))
print("Test shape:", X_test_scaled.shape)


Train mean after scaling: [0. 0.]
Test shape: (3, 2)


Remember:

> **FIT** only on training data. **TRANSFORM** both training and test data.

A `Pipeline` does this in a clean way.

In [94]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression()),
])

model.fit(X_train_s, y_train_s)
predictions = model.predict(X_test_s)
print("Predictions:", predictions)


Predictions: [1 0 1]


## 6. Data Splitting

After we prepare features, we split the data.

- **Training set** — the model learns from this.
- **Validation set** — we use this to compare models and settings.
- **Test set** — we use this only at the end, like new data.

In [95]:
# Example data for splitting
split_df = pd.DataFrame({
    "Age": [18, 19, 20, 21, 22, 23, 24, 25, 26, 27],
    "StudyHours": [1, 2, 3, 4, 5, 6, 7, 8, 2, 9],
    "Passed": [0, 0, 0, 0, 1, 1, 1, 1, 0, 1],
})

X = split_df.drop("Passed", axis=1)
y = split_df["Passed"]

print("X shape:", X.shape)
print("y counts:")
print(y.value_counts())


X shape: (10, 2)
y counts:
Passed
0    5
1    5
Name: count, dtype: int64


### 6.1 Train/Test Split

A common split is:

```
80% -> Training
20% -> Testing
```

In [96]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))


Train rows: 8
Test rows: 2


- `test_size=0.2` means 20% test data.
- `random_state=42` makes the split the same every time.

### 6.2 Train/Validation/Test Split

Sometimes we also need a validation set.

```
60% -> Training
20% -> Validation
20% -> Testing
```

In [97]:
# First, take 20% for testing
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# From the remaining 80%, take 25% for validation
# 0.25 * 0.80 = 0.20
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))


Train: 6
Validation: 2
Test: 2


### 6.3 Stratified Splitting

If classes are not balanced, use `stratify=y`.

This keeps a similar class ratio in train and test.

Example:

- Original: class 0 = 80%, class 1 = 20%
- After stratified split, train and test stay close to this ratio.

In [98]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Full data:")
print(y.value_counts(normalize=True).round(2))
print()
print("Train:")
print(y_train.value_counts(normalize=True).round(2))
print()
print("Test:")
print(y_test.value_counts(normalize=True).round(2))


Full data:
Passed
0    0.5
1    0.5
Name: proportion, dtype: float64

Train:
Passed
1    0.5
0    0.5
Name: proportion, dtype: float64

Test:
Passed
0    0.5
1    0.5
Name: proportion, dtype: float64


### 6.4 K-Fold Cross-Validation

In **K-Fold Cross-Validation**, we split the data into K parts.

Example: K = 5

The model trains 5 times. Each time, one part is for checking. The other parts are for training.

This gives a more stable result, especially with small datasets.

In [99]:
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression()),
])

scores = cross_val_score(cv_model, X, y, cv=kf)
print("Fold scores:", scores.round(3))
print("Mean score:", round(scores.mean(), 3))


Fold scores: [0.5 1.  1.  0.5 1. ]
Mean score: 0.8


There is no one best split for every project.

| Dataset size | Common choice |
|--------------|---------------|
| Small | Train / Validation / Test, or K-Fold |
| Medium | 70-80% train, 20-30% test |
| Large | 80-90% train, 10-20% test |

## 7. Complete Preprocessing Workflow

A simple learning order is:

```
1. Collect data
2. Check the data
3. Handle missing values
4. Remove duplicates
5. Handle outliers
6. Encode categories
7. Split the data
8. Scale numbers
9. Train the model
```

**Safer order in real work**

Split **before** you learn filling values, encodings, or scaling numbers.

```
Raw data
  -> Basic checks and duplicates
  -> Train/Test split
  -> Fit preprocessing on training data
  -> Transform train and test
  -> Train model
```

This helps avoid data leakage.

Below is a simple full example with the students data.

In [100]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# 1. Load data
df = pd.read_csv("../data/students.csv")

# 2. Basic checks
print("Shape:", df.shape)
print(df.isnull().sum())

# 3. Remove duplicates and unused text
df = df.drop_duplicates()
df = df.drop(columns=["Name"])

# 4. Split first
X = df.drop(columns=["Target"])
y = df["Target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_cols = ["Age", "Salary"]
ordinal_cols = ["Education"]
nominal_cols = ["City"]

education_order = [["High School", "Bachelor", "Master", "PhD"]]

# 5. Preprocess: fill, encode, scale
# Fit happens on training data only
preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_cols),
        ("edu", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(categories=education_order)),
        ]), ordinal_cols),
        ("city", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), nominal_cols),
    ]
)

workflow = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(random_state=42)),
])

workflow.fit(X_train, y_train)
print("Train accuracy:", round(workflow.score(X_train, y_train), 3))
print("Test accuracy:", round(workflow.score(X_test, y_test), 3))


Shape: (21, 6)
Name         0
Age          1
Salary       0
City         0
Education    0
Target       0
dtype: int64
Train accuracy: 0.75
Test accuracy: 0.75


**What this final example shows**

- We check the data first.
- We remove duplicates.
- We split before learning preprocessing.
- We fill numbers with the median.
- We fill City with the mode.
- We encode Education with order.
- We encode City with One-Hot Encoding.
- We scale numbers.
- We fit only on training data.

This is a learning pipeline. It is not a full production system.

If you remember one rule from this notebook, remember this:

> Learn from training data. Then apply the same steps to test data.